In [1]:
# Setup
import sys
import json
import time
import requests
import redis
from pathlib import Path

# Project root
current_dir = Path.cwd()
if current_dir.name == "module7" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
else:
    project_root = Path("/mnt/data/Portfolio/RAG")

sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

API = "http://localhost:8000/api/v1"

# Health check
print("\nMODULE 7 HEALTH CHECK")
print("=" * 40)

services = {
    "FastAPI" : "http://localhost:8000/api/v1/health",
    "OpenSearch": "http://localhost:9200/_cluster/health",
    "Ollama"  : "http://localhost:11434/api/version",
    "Langfuse": "http://localhost:3000/api/public/health",
}

for name, url in services.items():
    try:
        r = requests.get(url, timeout=5)
        print(f"{'✓' if r.status_code == 200 else '✗'} {name}")
    except Exception as e:
        print(f"✗ {name}: {e}")

try:
    redis.Redis(host="localhost", port=6379, socket_connect_timeout=3).ping()
    print("✓ Redis")
except Exception as e:
    print(f"✗ Redis: {e}")

Project root: /mnt/data/Portfolio/RAG

MODULE 7 HEALTH CHECK
✓ FastAPI
✓ OpenSearch
✓ Ollama
✓ Langfuse
✓ Redis


## Multi-Turn Dialogue

Every `/ask` and `/stream` request now supports an optional `session_id`.

| | Stateless | With `session_id` |
|---|---|---|
| History loaded | — | ✓ last 5 turns |
| Exact-match cache | ✓ | — |
| `session_id` in response | new UUID | echoed back |

Pass the returned `session_id` in the next request to continue the conversation.

In [2]:
# Step 1 — First request (no session_id)
print("STEP 1 — STATELESS REQUEST")
print("=" * 40)
print("No session_id sent. The API generates one and returns it.\n")

resp = requests.post(f"{API}/ask", json={
    "query"     : "What is retrieval-augmented generation?",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
}, timeout=120)

data = resp.json()
session_id = data["session_id"]

print(f"session_id : {session_id}")
print(f"chunks_used: {data['chunks_used']}")
print(f"cached     : {data.get('cached', False)}")
print(f"\nAnswer:\n{data['answer'][:500]}...")

STEP 1 — STATELESS REQUEST
No session_id sent. The API generates one and returns it.

session_id : 178e3183463d4d2eba7d58f6a8d25b03
chunks_used: 3
cached     : False

Answer:
Retrieval-augmented generation refers to the process of incorporating retrieved information into the generation process when it is not explicitly available. This technique can be particularly useful in tasks such as text summarization or question answering, where having access to relevant information can significantly enhance the quality of the generated output. In the context of language models like those used in LLMs, retrieval-augmented generation involves using techniques such as named entit...


In [3]:
# Step 2 — Follow-up (pass session_id)
print("STEP 2 — FOLLOW-UP REQUEST")
print("=" * 40)
print(f"Reusing session_id: {session_id}")
print("The model now sees the previous Q&A as context.\n")

resp2 = requests.post(f"{API}/ask", json={
    "query"     : "Can you give a concrete example of how it works?",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
    "session_id": session_id,
}, timeout=120)

data2 = resp2.json()
print(f"session_id echoed: {data2['session_id'] == session_id}")
print(f"\nAnswer:\n{data2['answer'][:500]}...")

STEP 2 — FOLLOW-UP REQUEST
Reusing session_id: 178e3183463d4d2eba7d58f6a8d25b03
The model now sees the previous Q&A as context.

session_id echoed: True

Answer:
The process involves several steps:

* Models are trained on large datasets to generate text based on certain prompts or personas. In this case, the Default Persona generates texts without system prompts, Classic Sydney uses the original Bing Search prompt, and Memetic Sydney is prompted with "You are Sydney". [1]
* These models then generate a corpus of LLM-generated texts on relationships between humans and AI, using a technique called memetic transfer.
* The corpus (named AI Sydney) contains ...


In [4]:
# Step 3 — Full conversation (3 turns)
print("STEP 3 — FULL CONVERSATION")
print("=" * 40)

questions = [
    "What are the main components of a RAG system?",
    "Which component has the biggest impact on answer quality?",
    "How would you improve that component?",
]

conv_session = None

for i, question in enumerate(questions, 1):
    payload = {
        "query"     : question,
        "top_k"     : 3,
        "use_hybrid": True,
        "model"     : "llama3.2:1b",
    }
    if conv_session:
        payload["session_id"] = conv_session

    r = requests.post(f"{API}/ask", json=payload, timeout=120)
    d = r.json()
    conv_session = d["session_id"]

    print(f"[Turn {i}] Q: {question}")
    print(f"         A: {d['answer'][:200]}...")
    print()

print(f"Session ID used throughout: {conv_session}")

STEP 3 — FULL CONVERSATION
[Turn 1] Q: What are the main components of a RAG system?
         A: A RAG system is composed of three key components: **Role**, **Action**, and **Goal**. These components are used to define the structure and behavior of a role-playing game-like system, where agents in...

[Turn 2] Q: Which component has the biggest impact on answer quality?
         A: The component with the biggest impact on answer quality is not explicitly stated in the papers. However, the authors of "Importance of Prompt Optimisation for Error Detection in Medical Notes Using La...

[Turn 3] Q: How would you improve that component?
         A: Analyzing the code provided by Sydney Telling, it appears that Memetic Sydney is designed to simulate a persona that originated from Microsoft's Bing Search platform. To improve this component, resear...

Session ID used throughout: 88d22fcec29a4a53bb013cd785e0e52e


In [5]:
# Step 4 — Streaming with session
print("STEP 4 — STREAMING WITH SESSION")
print("=" * 40)
print(f"Continuing conversation (session: {conv_session[:8]}...)\n")

stream_resp = requests.post(f"{API}/stream", json={
    "query"     : "Summarise what we have discussed so far.",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
    "session_id": conv_session,
}, stream=True, timeout=120)

print("Streaming answer:")
print("-" * 40)

full_answer = ""
returned_session = None

for line in stream_resp.iter_lines():
    if not line:
        continue
    text = line.decode("utf-8")
    if not text.startswith("data: "):
        continue
    event = json.loads(text[6:])

    if "chunk" in event:
        print(event["chunk"], end="", flush=True)
        full_answer += event["chunk"]

    if event.get("done"):
        returned_session = event.get("session_id")
        break

print(f"\n" + "-" * 40)
print(f"session_id consistent: {returned_session == conv_session}")

STEP 4 — STREAMING WITH SESSION
Continuing conversation (session: 88d22fce...)

Streaming answer:
----------------------------------------
So we've just started discussing something about IoT DDoS detection. We're talking about the challenges of protecting internet-connected devices from malicious attacks like distributed denial-of-service (DDoS) attacks, which can bring down entire networks or services. Our discussion is centered around exploring ways to improve the reliability and explainability of transfer learning models for this task.

We've also touched on a few papers and research studies that have investigated various aspects of IoT DDoS detection, including performance metrics, reliability-oriented statistics, latency and training cost assessment, and interpretability evaluation using techniques like Grad-CAM and SHAP. The goal is to identify the key factors that contribute to the success or failure of these models in real-world scenarios.

Lastly, we've had a brief mention of

In [6]:
# Step 5 — Inspect Redis session history
print("STEP 5 — SESSION HISTORY IN REDIS")
print("=" * 40)

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
key = f"session:{conv_session}:history"
ttl = r.ttl(key)
raw = r.get(key)

if raw:
    history = json.loads(raw)
    print(f"Key     : {key}")
    print(f"TTL     : {ttl}s (~{ttl // 3600}h {(ttl % 3600) // 60}m remaining)")
    print(f"Messages: {len(history)} ({len(history) // 2} turns)")
    print()
    for msg in history:
        role = "You  " if msg["role"] == "user" else "Model"
        print(f"  [{role}] {msg['content'][:120]}")
else:
    print("No history found — check session_id or Redis connection")

STEP 5 — SESSION HISTORY IN REDIS
Key     : session:88d22fcec29a4a53bb013cd785e0e52e:history
TTL     : 86400s (~24h 0m remaining)
Messages: 8 (4 turns)

  [You  ] What are the main components of a RAG system?
  [Model] A RAG system is composed of three key components: **Role**, **Action**, and **Goal**. These components are used to defin
  [You  ] Which component has the biggest impact on answer quality?
  [Model] The component with the biggest impact on answer quality is not explicitly stated in the papers. However, the authors of 
  [You  ] How would you improve that component?
  [Model] Analyzing the code provided by Sydney Telling, it appears that Memetic Sydney is designed to simulate a persona that ori
  [You  ] Summarise what we have discussed so far.
  [Model] So we've just started discussing something about IoT DDoS detection. We're talking about the challenges of protecting in


In [7]:
# Step 6 — New session (fresh context)
print("STEP 6 — NEW SESSION (FRESH CONTEXT)")
print("=" * 40)
print("Sending the same follow-up WITHOUT session_id.")
print("The model has no prior context and will ask for clarification.\n")

fresh_resp = requests.post(f"{API}/ask", json={
    "query"     : "Summarise what we have discussed so far.",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
    # no session_id
}, timeout=120)

fresh_data = fresh_resp.json()
new_session = fresh_data["session_id"]

print(f"Old session : {conv_session}")
print(f"New session : {new_session}")
print(f"Same session: {new_session == conv_session}")
print(f"\nAnswer (no context):\n{fresh_data['answer'][:400]}...")

STEP 6 — NEW SESSION (FRESH CONTEXT)
Sending the same follow-up WITHOUT session_id.
The model has no prior context and will ask for clarification.

Old session : 88d22fcec29a4a53bb013cd785e0e52e
New session : e1bc5e25801c4dbaa279b75b5c0ab5be
Same session: False

Answer (no context):
We've had a productive conversation so far. We started discussing the topic of IoT devices, specifically how they can be vulnerable to Distributed Denial-of-Service (DDoS) attacks under certain conditions. We also explored transfer learning models that have been used for IoT DDoS detection, highlighting their strengths and weaknesses in terms of performance, reliability, and interpretability.

We ...


In [8]:
# Summary
print("MODULE 7 SUMMARY")
print("=" * 40)

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
session_keys = r.keys("session:*:history")
cache_keys   = r.keys("exact_cache:*")

print(f"Active sessions in Redis : {len(session_keys)}")
print(f"Exact-match cache entries: {len(cache_keys)}")
print()
print("What we covered:")
print("  ✓ Stateless request returns a new session_id")
print("  ✓ Passing session_id gives the model conversation context")
print("  ✓ /stream supports session_id identically to /ask")
print("  ✓ History is stored in Redis with a 24-hour TTL")
print("  ✓ A new session starts with no prior context")
print("  ✓ Exact-match cache is bypassed for session requests")

MODULE 7 SUMMARY
Active sessions in Redis : 5
Exact-match cache entries: 5

What we covered:
  ✓ Stateless request returns a new session_id
  ✓ Passing session_id gives the model conversation context
  ✓ /stream supports session_id identically to /ask
  ✓ History is stored in Redis with a 24-hour TTL
  ✓ A new session starts with no prior context
  ✓ Exact-match cache is bypassed for session requests
